# 12 — Astra spatial copilot: question → plan → map

**Atlas source reading copy · browser-network**

[Course guide](index.ipynb) · [Pinned original](https://github.com/jltobias/JupyterLite-GeoLibre-GeoAI/blob/777fa8965068b1e26b982162f17254ee2df7ec84/book/notebooks/12_astra_spatial_copilot.ipynb)



Remote services, tiles, CDN assets or package retrieval are subject to network/CORS availability. Widget/anywidget/LexCube branches need their source extensions; the Atlas shell supplies only Pyodide.

**Provenance:** `book/notebooks/12_astra_spatial_copilot.ipynb` at `777fa8965068b1e26b982162f17254ee2df7ec84`; original SHA-256 `75e0c1a9bfeb4204dea30d2b93f2f3d4a506aa4fa1ebb3d2d7b56ffc6c75640a`. Original author material incorporated at repository owner request; no general license inferred. Third-party terms remain separate.

**Verification:** imported by static inspection; saved upstream outputs cleared. Runtime compatibility is separate from executed verification. See the course guide for dependencies, excluded assets, aliases and original credits.


# 12 — Astra spatial copilot: question → plan → map

[Launch in JupyterLite](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/lite/lab/index.html?path=12_astra_spatial_copilot.ipynb)

Turn a precise question into a constrained spatial analysis plan. Compare a choropleth, a scatter plot and a sensitivity curve; then verify a model plan against explicit operations. **Time:** about 20 minutes. **Prerequisites:** basic pandas; labs 00 and 02 are helpful.

**Runtime:** browser Python for the analysis; hosted GeoLibre for maps; optional CPython runner for Astra. The default run makes no model calls. The first run needs network access for packages, GeoLibre and interactive Plotly scripts. Static plots and local analysis do not depend on data APIs.

**Learning cycle:** predict → run → compare views → explain → test a different assumption.

## Start here: the question behind the code

**How can a copilot propose a spatial selection we can verify?**

**What you will learn:** ask → validate → inspect → challenge. The numbered reading guides below follow the actual code cells. Read a guide, predict the output, then run its cell with **Shift+Enter**. Run from top to bottom; changing an upstream value requires rerunning the cells that use it.

**GeoAI connection:** Bounded AI handoff: the default plan is hand-authored; optional model requests run through a separate full-Python runner.

**Why this is useful:** Keeping the operation explicit makes a proposed answer testable against table values and mapped geometry.

**Where the work happens:** Synthetic data, plan validation, filtering, linked charts and exports run in Pyodide. The default run sends no model request.

![A question becomes a checked operation. Start with a question that names a field, unit and threshold. Allow only defined fields and comparisons rather than executing generated Python. Link the same selected records to map outlines and charts. Move a threshold and observe how sensitive the selection is to the question.](assets/learning/12/infographic.svg)

### Watch the idea unfold

[[Source image continuation: Preview: A question becomes a checked operation — 32-second captioned walkthrough](https://github.com/jltobias/JupyterLite-GeoLibre-GeoAI/blob/777fa8965068b1e26b982162f17254ee2df7ec84/book/notebooks/assets/learning/12/poster.png)](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/lite/files/assets/learning/12/watch.html)

[▶ Watch the captioned mini-video and read its transcript](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/lite/files/assets/learning/12/watch.html) · [Launch this notebook](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/lite/lab/index.html?path=12_astra_spatial_copilot.ipynb) · [Read this Book chapter](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/notebooks/12_astra_spatial_copilot.html) · [Python and browser learning guide](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/learning-guide.html)

The video is a silent four-step teaching storyboard with player controls and captions. Open the companion page to play, pause, seek or download it. The local copy is bundled under `assets/learning/12/`; downloaded media can be played offline. The infographic and the text below cover the same ideas without a video player.

**Video transcript:**

1. **Ask (0–8 s):** Start with a question that names a field, unit and threshold. Code landmark: `temperature >= 30`.

2. **Validate (8–16 s):** Allow only defined fields and comparisons rather than executing generated Python. Code landmark: `apply_plan(table, plan)`.

3. **Inspect (16–24 s):** Link the same selected records to map outlines and charts. Code landmark: `selection_ids = set(selected.id)`.

4. **Challenge (24–32 s):** Move a threshold and observe how sensitive the selection is to the question. Code landmark: `update_threshold(threshold)`.


## See the connections

![Astra spatial copilot: plans become evidence: Question: A precise spatial request defines a test; Astra: A structured plan chooses a bounded operation; Python + GeoLibre: Compute, map and compare the selected cells; Check: Validate the plan and ground every claim](assets/12_mindmap.svg)

### Code guide 1 — Prepare the browser toolbox

**Read the Python:** `import` makes a package available to Python. `sys.platform` identifies the runtime; the indented `if` block runs only in Pyodide. `await` waits for package loading. `LiteMap as Map` gives the browser-compatible map class a shorter name. Installing a package and importing its functions are separate steps.

**Look for:** Imports complete before you run the next cell. The first run can take longer while packages download.

**Pause and try:** Find the alias `Map`. If a later cell says a name is undefined, rerun this setup cell first.


In [ ]:
import sys
if sys.platform == "emscripten":
    from pyodide import loadPackage
    await loadPackage(["numpy", "pandas", "matplotlib", "scipy", "pillow"])
    import micropip
    await micropip.install(["geolibre==3.0.0", "plotly==6.5.0"])
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from geolibre_lite import LiteMap as Map
from cognition import style_plots, COLORS, save_json
from astra_lab import make_request, response_text
style_plots()

## 1. Ask a question that code can answer

**Question:** which teaching grid cells have temperature at least 30 °C? This is a selection exercise, not a heat-risk assessment. All temperatures, canopy fractions and population counts are invented.

Before running, predict whether low canopy alone will explain every warmer cell. The generator contains a separate spatial heat bump, so the views should disagree with that simple explanation.

### Code guide 2 — Start with a known synthetic city

**Read the Python:** The helper returns two aligned representations: a table of invented attributes and a GeoJSON grid. `head()` previews rows; `unique()` confirms the source label. Keeping stable cell IDs lets later views select the same places.

**Look for:** 64 teaching cells, each with invented temperature, canopy and population fields.

**Pause and try:** Locate one cell ID and explain how it will connect the table to a map polygon.


In [ ]:
from cognition import teaching_city
table, city = teaching_city()
display(table.head())
print("Cells:", len(table), "source:", table.source.unique().tolist())

## 2. Let a plan choose only a permitted operation

The offline plan below is a **hand-authored example**, not GPT output. `apply_plan` accepts only two named fields, two comparisons and finite thresholds. A live model plan passes through the same checks. No model-written Python is executed.

### Code guide 3 — Execute a bounded plan and test sensitivity

**Read the Python:** The dictionary states a field, comparison and threshold. `apply_plan` validates that allowed operation and filters rows. A set stores selected IDs. The panels compare the temperature grid, canopy scatter and percentage selected as the threshold changes.

**Look for:** A repeatable selection from a hand-authored plan; the default code has not asked a language model to choose it.

**Pause and try:** Increase the threshold from 30 to 32. Predict the direction of the selected-count change.


In [ ]:
from astra_lab import PLAN_SCHEMA, apply_plan
plan = {"field": "temperature_c", "operator": ">=", "threshold": 30.0,
        "reason": "Find cells that meet the explicit 30 C teaching threshold."}
selected = apply_plan(table, plan)
selection_ids = set(selected.id)
print(f"Selected {len(selected)} of {len(table)} cells")
fig, axes = plt.subplots(1, 3, figsize=(14, 4), layout="constrained")
heat = table.temperature_c.to_numpy().reshape(8, 8)
im = axes[0].imshow(heat, origin="lower", cmap="magma", extent=(0,1.6,0,1.6))
axes[0].set(xlabel="Local east (km)", ylabel="Local north (km)", title="Synthetic temperature")
fig.colorbar(im, ax=axes[0], label="°C")
axes[1].scatter(table.canopy_pct, table.temperature_c, c="#94a3b8", label="Other cells")
axes[1].scatter(selected.canopy_pct, selected.temperature_c, c="#d97706", label="Selected")
axes[1].axhline(plan['threshold'], color="#475569", ls="--")
axes[1].set(xlabel="Invented canopy (%)", ylabel="Invented temperature (°C)", title="Does one feature explain the pattern?")
axes[1].legend(fontsize=8)
thresholds = np.linspace(26,35,37)
shares = [100 * table.temperature_c.ge(t).mean() for t in thresholds]
axes[2].plot(thresholds, shares)
axes[2].axvline(30, color="#d97706", ls="--")
axes[2].set(xlabel="Temperature threshold (°C)", ylabel="Selected cells (%)", title="Sensitivity to the question")
plt.show()

### Code guide 4 — Map and export the selected cells

**Read the Python:** The list comprehension keeps polygons whose IDs are in the selected set. The conditional avoids drawing an empty overlay. `save_json` writes a GeoJSON subset and map recipe to the notebook filesystem for download.

**Look for:** A temperature map with selected outlines and two export files.

**Pause and try:** Check a boundary cell's value against the >= comparison rather than guessing from its color.


In [ ]:
city_map = Map(center=(-122.291,37.907), zoom=13, height="550px")
city_map.add_choropleth(city, "temperature_c", name="Synthetic temperature (°C)", colormap="magma")
selected_fc = {"type":"FeatureCollection", "features":[f for f in city['features'] if f['properties']['id'] in selection_ids]}
if selected_fc['features']:
    city_map.add_geojson(selected_fc, name="Selected: at least 30 °C", fillOpacity=.05,
                        strokeColor="#22d3ee", strokeWidth=3)
display(city_map)
save_json("exports/selected_cells.geojson", selected_fc)
save_json("exports/teaching_city.geolibre.json", city_map.to_project())

### Change the threshold and watch both views

Move the slider, release it, and compare the selected cells with the distribution. The orange cells on the map and the count in the chart use the same rule. This control explores sensitivity; the Astra request below still asks the explicit 30 °C question.

### Code guide 5 — Connect one slider to two views

**Read the Python:** The callback runs when the slider changes. It updates the threshold, recomputes IDs, resets a deep copy of the base project and draws a histogram. `continuous_update=False` waits until you release the slider, reducing redraw work.

**Look for:** A linked selection overlay and histogram with a moving threshold line.

**Pause and try:** Drag to 34 and back to 30. Why would failing to reset the project accumulate old layers?


In [ ]:
import copy
import ipywidgets as widgets
linked_map = Map(center=(-122.291,37.907),zoom=13,height="450px")
linked_map.add_geojson(city,name="All synthetic cells",fillColor="#cbd5e1",fillOpacity=.25,strokeWidth=.5)
base_project = copy.deepcopy(linked_map.project)
slider = widgets.FloatSlider(value=30,min=26,max=35,step=.25,description="Threshold °C",
                             continuous_update=False,style={"description_width":"initial"})
def update_threshold(threshold):
    picked = apply_plan(table,{**plan,"threshold":threshold})
    ids = set(picked.id)
    linked_map.project = copy.deepcopy(base_project)
    if ids:
        linked_map.add_geojson({"type":"FeatureCollection","features":[f for f in city['features'] if f['properties']['id'] in ids]},
                              name=f"At least {threshold:g} °C",fillColor="#d97706",fillOpacity=.85,strokeWidth=.7)
    fig,ax=plt.subplots(figsize=(9,3),layout="constrained")
    ax.hist(table.temperature_c,bins=np.arange(25,36,.5),color="#94a3b8")
    ax.axvline(threshold,color="#d97706",lw=3)
    ax.set(xlabel="Synthetic temperature (°C)",ylabel="Cells",
           title=f"{len(picked)} / {len(table)} cells meet the threshold")
    plt.show()
linked_chart = widgets.interactive_output(update_threshold,{"threshold":slider})
display(widgets.VBox([slider,linked_chart]),linked_map)

## 3. Prepare a real Astra structured-output request

Astra receives the question, units, a compact table and the allowed plan schema. Geography is already attached to stable cell IDs. Change the question to “canopy at most 25%” to try a different valid plan; after importing a response, recompute the selected map with the same function.

### Code guide 6 — Package evidence for an optional copilot

**Read the Python:** The evidence dictionary includes units, grid size, source and records. `make_request` builds a structured request with an allowed output schema. Saving that JSON prepares a handoff; it does not make an API call.

**Look for:** A request file that can be inspected before using the separate full-Python runner.

**Pause and try:** Find the source and units fields that prevent a model from mistaking this fixture for measured city temperatures.


In [ ]:
evidence = {"evidence_id":"synthetic_city", "crs":"EPSG:4326 for display; local grid in metres",
            "cell_size_m":200, "source":"synthetic teaching fixture", "rows":table.to_dict('records')}
request = make_request("Select cells with temperature at least 30 degrees C. Return a bounded plan.",
                       evidence, schema=PLAN_SCHEMA)
save_json("exports/spatial_plan_request.json", request)
print("Prepared", request['model'], "request; no API call made.")

## Optional: run this request with GPT-6 Astra

All calculations above run locally. The exported request has **not been sent** and any example plan is **hand-authored**, not a recorded model response. To obtain a live response:

1. Download the generated request from JupyterLite's file browser (right-click → Download).
2. In a full CPython terminal in this repository, set `OPENAI_API_KEY` in that terminal's environment. Never put the key in a notebook, output, or the static site.
3. Preview with `python scripts/run_astra.py PATH_TO_REQUEST.json`. Send deliberately with `--send --output astra_response.json`. This uses your API account and can incur charges.
4. Upload `astra_response.json` to the notebook folder, set the import path in the next cell, and rerun it. Inspect its evidence claims before using them.

For notebook 15, also download `accessibility_evidence.json` and pass `--evidence PATH_TO_EVIDENCE.json`. The runner only dispatches the one allowlisted local tool, with a bounded number of rounds.

The static JupyterLite site does not host this runner or proxy requests. Model access depends on your API project. Image reading is an aid to interpretation; geometry, distances and metrics come from code and explicit metadata.

### Code guide 7 — Inspect an optional model response

**Read the Python:** `None` means no response file was supplied. Only a real path enters the `if` branch; `json.loads` converts saved JSON text into Python objects. The `else` branch keeps the default lesson usable without an API call. Follow the preceding export/import instructions for live use.

**Look for:** The default run prints an offline message. An imported answer is evidence to inspect, not a new measurement.

**Pause and try:** Write your own explanation first, then check any imported response against the computed values and units.


In [ ]:
RESPONSE_PATH = None  # e.g. "astra_response.json" after uploading an actual response
if RESPONSE_PATH:
    actual_plan = json.loads(response_text(json.loads(Path(RESPONSE_PATH).read_text())))
    actual_selection = apply_plan(table, actual_plan)
    display(actual_selection)
    actual_fc = {"type":"FeatureCollection", "features":[f for f in city['features'] if f['properties']['id'] in set(actual_selection.id)]}
    if actual_fc['features']:
        actual_map = Map(center=(-122.291,37.907),zoom=13,height="450px")
        actual_map.add_geojson(actual_fc,name="Validated Astra selection",fillColor="#0f766e")
        display(actual_map)
    else:
        print("The validated plan selected no cells.")
    print("Matches hand-authored baseline IDs:", set(actual_selection.id) == selection_ids)
else:
    print("Offline teaching run. No Astra response imported.")

## 4. Transfer the pattern to real GeoAI

Use lab 02's county features or lab 08's STAC footprints in place of the synthetic grid. Retain units, source IDs, date and geometry. Let Astra propose a bounded operation; let Python perform it; compare the result with a baseline and inspect it in GeoLibre.

**Explain:** why can a perfectly valid JSON plan still misunderstand a question? Try a threshold beyond the physical range and verify that validation rejects it. Then change a valid threshold and describe the selection curve without implying that cell share equals population share.

## Data & software citations

- Teaching fixtures: deterministic synthetic arrays and networks authored in this repository. Values, scenes, facilities and outcomes are invented; coordinates only anchor the display. No actual people, environmental measurements, or service areas are represented.
- [GeoLibre](https://geolibre.app/) and [GeoLibre map controls](https://geolibre.app/user-guide/map-controls/); `geolibre==3.0.0` project builders through `geolibre_lite.LiteMap`.
- [GeoAI](https://opengeoai.org/): full Python/GPU continuation for imagery models. These browser labs implement workflow patterns; they do not install or run `geoai-py`.
- [GPT-6 Astra model](https://developers.openai.com/api/docs/models/gpt-6-astra), [Structured Outputs](https://developers.openai.com/api/docs/guides/structured-outputs), [image inputs](https://developers.openai.com/api/docs/guides/images-vision), and [function calling](https://developers.openai.com/api/docs/guides/function-calling). API examples checked 2026-10-02.
- [Pyodide packages](https://pyodide.org/en/stable/usage/packages-in-pyodide.html), [Matplotlib](https://matplotlib.org/stable/), [Plotly Python](https://plotly.com/python/).
- GeoLibre basemap tiles retain their provider attribution, including [OpenStreetMap contributors](https://www.openstreetmap.org/copyright) where applicable.

See `DATA_SOURCES.md` for the source register and runtime boundaries.

## Make sense of the result

![The copilot says these cells are hot. Show the field, units, comparison and threshold. Now I can reproduce and challenge the selection.](assets/learning/12/cartoon.svg)

**Cartoon transcript:** The copilot says these cells are hot. → Show the field, units, comparison and threshold. → Now I can reproduce and challenge the selection.

**Explain it to a colleague:** What would convince you that a live plan matches the hand-authored baseline?

<details><summary>Check your reasoning</summary>

Validate the operation, recompute it and compare selected cell IDs, not just the wording of the answer.

</details>

Write three sentences: **I used…** (source and units), **the code computed…** (method), and **I can conclude…** (plus a remaining question). Point to the map, chart or table that supports each claim.

All new teaching illustrations are original schematics. Their marks and fictional dialogue are explanatory, not observations, fitted predictions or captured AI conversations. The existing citation cells identify the analytical data.


## GeoLibre creator and software attribution

GeoLibre is developed by **Qiusheng Wu and contributors**. Recommended citation: Wu, Q. (2026). *GeoLibre: A lightweight, cloud-native GIS platform for visualizing, exploring, and analyzing geospatial data.* Zenodo. [doi:10.5281/zenodo.20785400](https://doi.org/10.5281/zenodo.20785400). See [the official citation guidance](https://geolibre.app/citation/) and [source and MIT license](https://github.com/opengeos/GeoLibre). These browser lessons pin the Python builders to `geolibre==3.0.0`; the independently hosted viewer may change. This repository supplies the JupyterLite adapter and teaching materials, not the upstream GIS. Keep the original dataset, demo-author and basemap credits when sharing a result.
